<div style="background:linear-gradient(125deg,#071A2B,#123F4A);color:#F8F4E8;padding:40px;border-radius:20px;">
<p style="color:#F4C95D;letter-spacing:.17em;font-size:12px;">PLAYGROUND S6E9 · TEACHER → STUDENT LAB</p>
<h1 style="color:#F8F4E8;font-size:40px;line-height:1.08;">Pseudo-Labels Without Lying to CV</h1>
<p style="font-size:20px;">15 honest OOF teachers, ordinary customer data, one tiny rank-aware student.</p>
<p style="color:#39D0C3;">Keep 0.94656 safe · train the next idea · measure it before trusting it</p>
</div>

### What this notebook changes

The tempting idea is to train on the competition rows and “mix in the answers.” The true test answers do not exist. Treating public submissions as truth would copy leaderboard noise. Instead we use **OOF predictions as soft answers**: every training-row prediction was made by a model that did not train on that row.

1. Rebuild the proven 15-view L1 teacher on frozen 5-fold OOF.
2. Give a shallow LightGBM student the 15 teacher ranks **plus ordinary customer features**.
3. Blend only 20% student ranking into the teacher and require positive overall OOF with at least 4/5 fold wins.
4. Preserve the measured **0.94656** public ranking as `submission.csv`.
5. Export a cautious 3% trained perturbation separately as `submission_try_student_3pct.csv`.

The original 9,466 labelled EV rows are deliberately absent: our prior controlled experiment lost **0.000034 OOF and 0/5 folds** when they were added.

## 01 · Inputs and contracts

Attach four inputs: the competition, `najiama/s6e9-oof`, `megayak/s6e9-six-feature-views-oof-library`, and `jazivxt/s6e9-zoom-zoom-baseline`. The first three create the trained private-first model. The last one supplies the byte-pinned 0.94651 anchor used to reproduce the current 0.94656 public file.

In [ ]:
from pathlib import Path
import hashlib, io, json, time, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import rankdata, spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from IPython.display import HTML, display

SEED, ID, TARGET = 42, "id", "Will_Buy_EV"
ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
OUTPUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs/teacher-student")
OUTPUT.mkdir(parents=True, exist_ok=True)

def one(pattern, contains=None):
    hits = sorted({p for p in ROOT.rglob(pattern) if p.is_file()})
    if contains:
        filtered = [p for p in hits if contains.lower() in str(p).lower()]
        if filtered:
            hits = filtered
    if len(hits) != 1:
        raise RuntimeError(f"Expected one {pattern}; found {len(hits)}: {hits[:8]}")
    return hits[0]

def rank01(values):
    return pd.Series(np.asarray(values)).rank(method="average", pct=True).to_numpy()

print("Input root:", ROOT)

In [ ]:
train = pd.read_csv(one("train.csv", contains="playground-series-s6e9"))
test = pd.read_csv(one("test.csv", contains="playground-series-s6e9"))
sample = pd.read_csv(one("sample_submission.csv", contains="playground-series-s6e9"))
test = pd.DataFrame({ID: sample[ID]}).merge(test, on=ID, how="left", validate="one_to_one")
y = train[TARGET].map({"No": 0, "Yes": 1}) if train[TARGET].dtype == object else train[TARGET].astype(int)
fold_path = next(iter(sorted(ROOT.rglob("v1.parquet"))), None)
if fold_path is not None:
    fold_frame = pd.read_parquet(fold_path)
    folds = pd.DataFrame({ID: train[ID]}).merge(fold_frame[[ID, "fold"]], on=ID, validate="one_to_one")["fold"].to_numpy()
else:
    from sklearn.model_selection import StratifiedKFold
    folds = np.empty(len(train), dtype=np.int8)
    for fold, (_, valid) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(train, y)):
        folds[valid] = fold
assert train[ID].is_unique and sample[ID].is_unique and test[ID].equals(sample[ID])
assert set(np.unique(folds)) == set(range(5)) and y.isin([0, 1]).all()
print(f"train={train.shape} · test={test.shape} · positive={y.mean():.3%} · folds=v1/seed42")

## 02 · Fifteen teachers, one prediction per row

Six established OOF sources are joined with six alternative feature views, their ensemble, and two RealMLP views. Every file is aligned by `id`; labels, finiteness and uniqueness are asserted. Percentile ranks put differently calibrated models on the same ROC-AUC scale.

In [ ]:
CORE = {
    "01 Blend": ("01_blend_oof.csv", "01_submission.csv"),
    "Pure LGBM v1": ("Pure LGBM_V1_oof.csv", "Pure LGBM_V1_test.csv"),
    "Pure LGBM v3": ("Pure LGBM_V3_oof.csv", "Pure LGBM_V3_test.csv"),
    "Sergey LGBM": ("Sergey_LGBM_oof.csv", "Sergey_LGBM_submission.csv"),
    "XGB triple TE 10f": ("XGBoost_Triple_TE_10folds_oof.csv", "XGBoost_Triple_TE_10folds_test.csv"),
    "XGB triple TE 5f": ("XGBoost_Triple_TE_5folds_oof.csv", "XGBoost_Triple_TE_5folds_test.csv"),
}
SIX = [
    "A_lgbm_triple_te_digits_3seed", "B_xgb_on_A_features",
    "C_no_digits_windows_lift_sm2_30_300", "D_no_exact_key_ladder_windows",
    "E_ladder25_250_2500_lift_sm5_50_500", "F_exact_rate_as_init_score", "ensemble",
]
REAL = ["G_realmlp_10fold", "G_realmlp_3seed"]

def prediction(path, ids):
    frame = pd.read_csv(path)
    cols = [c for c in frame if c != ID]
    assert ID in frame and frame[ID].is_unique and len(cols) == 1
    out = pd.DataFrame({ID: ids}).merge(frame[[ID, cols[0]]], on=ID, how="left", validate="one_to_one")[cols[0]].to_numpy(float)
    assert np.isfinite(out).all() and len(out) == len(ids)
    return out

def aligned(frame, ids, column):
    assert ID in frame and frame[ID].is_unique and column in frame
    out = pd.DataFrame({ID: ids}).merge(frame[[ID, column]], on=ID, how="left", validate="one_to_one")[column].to_numpy(float)
    assert np.isfinite(out).all() and len(out) == len(ids)
    return out

oof, test_pred = {}, {}
for name, (of, tf) in CORE.items():
    oof[name], test_pred[name] = prediction(one(of), train[ID]), prediction(one(tf), sample[ID])

six_oof, six_test = pd.read_csv(one("oof_six_views.csv")), pd.read_csv(one("test_six_views.csv"))
real_oof, real_test = pd.read_csv(one("oof_realmlp_g.csv")), pd.read_csv(one("test_realmlp_g.csv"))
assert np.array_equal(six_oof[TARGET].astype(int).to_numpy(), y.to_numpy())
assert np.array_equal(real_oof[TARGET].astype(int).to_numpy(), y.to_numpy())
for name in SIX:
    oof[name], test_pred[name] = aligned(six_oof, train[ID], name), aligned(six_test, sample[ID], name)
for name in REAL:
    oof[name], test_pred[name] = aligned(real_oof, train[ID], name), aligned(real_test, sample[ID], name)

columns = list(oof)
rank_oof = pd.DataFrame({k: rank01(v) for k, v in oof.items()})
rank_test = pd.DataFrame({k: rank01(v) for k, v in test_pred.items()})
audit = pd.DataFrame({"teacher": columns, "OOF AUC": [roc_auc_score(y, oof[k]) for k in columns]})
display(audit.sort_values("OOF AUC", ascending=False).style.format({"OOF AUC": "{:.7f}"}).background_gradient(cmap="YlGnBu"))
assert rank_oof.shape == (len(train), 15) and rank_test.shape == (len(test), 15)

## 03 · Rebuild the private-first teacher

The teacher is the unchanged five-fold L1 stack from the previous notebook. Scaler and logistic regression fit inside each outer fold. The held fold receives exactly one prediction; five test predictions are averaged. This is the control the student must beat.

In [ ]:
X, XT = rank_oof.to_numpy(), rank_test.to_numpy()
teacher_oof, teacher_test = np.zeros(len(train)), np.zeros(len(test))
teacher_folds, coef_rows = [], []
started = time.perf_counter()
for fold in range(5):
    fit, valid = folds != fold, folds == fold
    scaler = StandardScaler().fit(X[fit])
    model = LogisticRegression(penalty="l1", solver="saga", C=1.0, max_iter=1200,
                               tol=1e-5, random_state=SEED + fold, n_jobs=-1)
    model.fit(scaler.transform(X[fit]), y.iloc[np.flatnonzero(fit)])
    teacher_oof[valid] = model.predict_proba(scaler.transform(X[valid]))[:, 1]
    teacher_test += model.predict_proba(scaler.transform(XT))[:, 1] / 5
    score = roc_auc_score(y.iloc[np.flatnonzero(valid)], teacher_oof[valid])
    teacher_folds.append(score)
    coef_rows.append(dict(fold=fold, **dict(zip(columns, model.coef_[0]))))
    print(f"teacher fold {fold}: {score:.7f}")
teacher_auc = roc_auc_score(y, teacher_oof)
print(f"teacher OOF={teacher_auc:.9f} · {time.perf_counter()-started:.1f}s")

## 04 · What the student can learn

The student receives two information blocks:

- **soft answers:** the 15 OOF ranks, which summarize different models and feature representations;
- **ordinary data:** age, income, commute, charging access, subsidy, anxiety and the remaining customer fields.

Categorical codes are learned inside each fold with unknown values mapped to `-1`. The tree is intentionally shallow (`5` leaves, at least `4,000` rows per leaf). Its job is a small nonlinear reorder, not memorization. We freeze a 20% rank weight from the local screen rather than tune dozens of weights on the final OOF.

In [ ]:
raw_columns = [c for c in test.columns if c != ID]
cat_columns = [c for c in raw_columns if train[c].dtype == object]
num_columns = [c for c in raw_columns if c not in cat_columns]

def raw_matrix(fit_idx, apply_idx, apply_test=False):
    fit_frame = train.iloc[fit_idx]
    apply_frame = test if apply_test else train.iloc[apply_idx]
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                             encoded_missing_value=-1, dtype=np.float32)
    encoder.fit(fit_frame[cat_columns].astype(str))
    left = apply_frame[num_columns].to_numpy(np.float32)
    right = encoder.transform(apply_frame[cat_columns].astype(str))
    return np.column_stack([left, right]).astype(np.float32)

student_oof, student_test = np.zeros(len(train)), np.zeros(len(test))
student_folds, best_iterations = [], []
for fold in range(5):
    fit_idx, valid_idx = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
    Xfit = np.column_stack([X[fit_idx].astype(np.float32), raw_matrix(fit_idx, fit_idx)])
    Xvalid = np.column_stack([X[valid_idx].astype(np.float32), raw_matrix(fit_idx, valid_idx)])
    Xtest_fold = np.column_stack([XT.astype(np.float32), raw_matrix(fit_idx, None, apply_test=True)])
    model = LGBMClassifier(
        objective="binary", n_estimators=700, learning_rate=0.02,
        num_leaves=5, min_child_samples=4000, colsample_bytree=0.75,
        reg_alpha=1.0, reg_lambda=20.0, random_state=SEED + fold,
        n_jobs=-1, verbosity=-1,
    )
    model.fit(Xfit, y.iloc[fit_idx], eval_set=[(Xvalid, y.iloc[valid_idx])], eval_metric="auc",
              callbacks=[early_stopping(60, verbose=False), log_evaluation(0)])
    student_oof[valid_idx] = model.predict_proba(Xvalid)[:, 1]
    student_test += model.predict_proba(Xtest_fold)[:, 1] / 5
    score = roc_auc_score(y.iloc[valid_idx], student_oof[valid_idx])
    student_folds.append(score); best_iterations.append(model.best_iteration_)
    print(f"student fold {fold}: {score:.7f} · trees={model.best_iteration_}")

## 05 · The only decision that matters: paired OOF

The student is expected to be weaker alone. Diversity is useful only if a small dose improves the same held rows. We blend percentile ranks inside each validation fold, so fold sizes cannot distort the comparison. Acceptance requires a positive full OOF delta and at least four positive fold deltas.

In [ ]:
WEIGHT = 0.20
candidate_oof = (1-WEIGHT)*rank01(teacher_oof) + WEIGHT*rank01(student_oof)
candidate_folds = [roc_auc_score(y[folds == fold], candidate_oof[folds == fold]) for fold in range(5)]
candidate_auc = roc_auc_score(y, candidate_oof)
fold_deltas = np.asarray(candidate_folds) - np.asarray(teacher_folds)
accepted = bool(candidate_auc > teacher_auc and (fold_deltas > 0).sum() >= 4)
private_test = ((1-WEIGHT)*rank01(teacher_test) + WEIGHT*rank01(student_test)) if accepted else rank01(teacher_test)

comparison = pd.DataFrame({"fold": range(5), "teacher": teacher_folds,
                           "teacher + student": candidate_folds, "delta": fold_deltas})
display(comparison.style.format({"teacher":"{:.7f}","teacher + student":"{:.7f}","delta":"{:+.7f}"})
        .background_gradient(subset=["delta"], cmap="RdYlGn"))
print(f"teacher={teacher_auc:.9f} · candidate={candidate_auc:.9f} · delta={candidate_auc-teacher_auc:+.9f}")
print(f"fold wins={(fold_deltas>0).sum()}/5 · accepted={accepted}")

## 06 · Keep the measured score, export the trained probe

`submission.csv` reproduces the measured 0.94656 ordering exactly. The trained OOF-selected model is saved as `submission_private_first.csv`. A deliberately small 3% rank dose into the public file is exported as the next leaderboard probe; it is never presented as OOF-validated because the public bands have no train-row counterpart.

In [ ]:
EXPECTED_SHA256 = "1f40cec214646ba50eac6d5307d41c6aca82fe7acc38e2501399980c62d4bbe7"
catalog = pd.read_csv(one("ranked_catalog_latest.csv", contains="zoom-zoom-baseline"), dtype={"sha256": str})
archive_path = one("ranked_predictions_latest.zip.bin", contains="zoom-zoom-baseline")
match = catalog[catalog.sha256.str.lower() == EXPECTED_SHA256]
assert len(match) == 1
with zipfile.ZipFile(archive_path) as archive:
    payload = archive.read(match.iloc[0].csv_path)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256
anchor = pd.read_csv(io.BytesIO(payload), float_precision="round_trip")
assert anchor[ID].equals(sample[ID]) and anchor.columns.tolist() == [ID, TARGET]

BANDS = [(0.54, 0.04), (0.20, 0.06), (0.58, 0.04), (0.47, 0.06)]
base = rank01(anchor[TARGET].to_numpy(float)); adjusted = base.copy()
for start, width in BANDS:
    half = width/2
    lower = (base >= start) & (base < start+half)
    upper = (base >= start+half) & (base < start+width)
    adjusted[lower] += half; adjusted[upper] -= half
public_safe = rankdata(adjusted, method="ordinal") / len(adjusted)
changed = int(np.count_nonzero(rankdata(base, method="ordinal") != rankdata(public_safe, method="ordinal")))
assert changed == 57314 and len(np.unique(public_safe)) == len(public_safe)

try_weight = 0.03
student_probe = (1-try_weight)*rank01(public_safe) + try_weight*rank01(private_test)
pd.DataFrame({ID: sample[ID], TARGET: public_safe}).to_csv(OUTPUT/"submission.csv", index=False)
pd.DataFrame({ID: sample[ID], TARGET: private_test}).to_csv(OUTPUT/"submission_private_first.csv", index=False)
pd.DataFrame({ID: sample[ID], TARGET: student_probe}).to_csv(OUTPUT/"submission_try_student_3pct.csv", index=False)
pd.DataFrame({ID: train[ID], "fold": folds, TARGET: y, "teacher": teacher_oof,
              "student": student_oof, "candidate": candidate_oof}).to_csv(OUTPUT/"oof_teacher_student.csv", index=False)
pd.DataFrame(coef_rows).to_csv(OUTPUT/"teacher_coefficients.csv", index=False)

report = {
    "metric":"roc_auc", "direction":"maximize", "fold_version":"v1", "seed":SEED,
    "teacher_auc":teacher_auc, "candidate_auc":candidate_auc,
    "delta":candidate_auc-teacher_auc, "fold_deltas":fold_deltas.tolist(),
    "fold_wins":int((fold_deltas>0).sum()), "accepted":accepted,
    "student_weight":WEIGHT, "public_safe_reported_lb":0.94656,
    "public_rank_changes":changed, "probe_weight":try_weight,
    "best_iterations":best_iterations,
}
(OUTPUT/"teacher_student_report.json").write_text(json.dumps(report, indent=2))
display(HTML(f'''<div style="background:#071A2B;color:#F8F4E8;padding:25px;border-radius:16px;">
<p style="color:#F4C95D;letter-spacing:.13em;">OUTPUT CONTRACT</p>
<h2 style="color:#F8F4E8;">submission.csv keeps the measured 0.94656</h2>
<b>OOF student accepted: {accepted}</b> · delta {candidate_auc-teacher_auc:+.9f} · wins {(fold_deltas>0).sum()}/5<br>
The new leaderboard probe is <code>submission_try_student_3pct.csv</code>.
</div>'''))
print(sorted(p.name for p in OUTPUT.iterdir()))

### What we learned

- Extra labelled source rows already failed a controlled 5-fold test, so more rows were not automatically more signal.
- Direct residual regression failed in the local screen on all five folds because probability error and ranking error are different objectives.
- The shallow student asks the right question: can nonlinear combinations of independent OOF ranks and raw context correct a few pairwise orderings?
- The current 0.94656 is protected. The 3% student file is a bounded experiment, while the full trained output remains available for private-leaderboard diversification.

If the 3% probe improves, test 5% once. If it ties or falls, stop this branch; repeated public probing would turn the leaderboard into the training set.